it is also called as the thread memory

The history for the frontend is managed in seperate backed. In thread memory we implement this for proper AI response maintaing context.

We properly store the context by using n previous messages and history of all other messages older than last n messages

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_core.messages import trim_messages, BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import RemoveMessage

In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",  # Your Azure deployment name
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1",
    #add your api key below
    # api_key=
    )

MAX_TOKENS = 150

In [ ]:
class State(TypedDict):
    Human: str
    AI: str
    summary: str
    messages: Annotated[list[BaseMessage], operator.add]

In [ ]:
def trimmer_node(state: State):
    trimmed_messags= trim_messages(
    messages=state["messages"],
    max_tokens=MAX_TOKENS,
    strategy="last",
    token_counter=llm,
)
    
    return {
    "messages": trim_messages
    }

    

In [ ]:
def summary_node(state : State):
    messages = state['messages']
    summary = state['summary']
    if summary:
        prompt = (
            f"Extend the summary {state['summary']} using the new messages: {state['messages']}"
        )
    else:
        prompt=(
            f"Generate summary of these messages {state['messages']}"
        )
    messages += HumanMessage(content=prompt)
    response = llm.invoke(messages)

    messages_to_delete = [
        RemoveMessage(id=msg.id)
        for msg in messages[:4]
        ]
    
    return {
        "messages":messages_to_delete,
        "summary": response.content
    }
    